In [2]:
import numpy as np
import pandas as pd

In [3]:
customer = pd.read_csv("./olist_customers_dataset.csv")
location=pd.read_csv("./olist_geolocation_dataset.csv")
order_item=pd.read_csv("./olist_order_items_dataset.csv")
order=pd.read_csv("./olist_orders_dataset.csv")
review=pd.read_csv("./olist_order_reviews_dataset.csv")
sellers=pd.read_csv("./olist_sellers_dataset.csv")
product=pd.read_csv("./olist_products_dataset.csv")

In [4]:
#  Step 1: Orders + Customers 
order_customer = pd.merge(order, customer, on="customer_id", how="left")

#  Step 2: Add Order Items 
order_customer_item = pd.merge(order_customer, order_item, on="order_id", how="left")

#  Step 3: Add Sellers 
order_customer_item_seller = pd.merge(order_customer_item, sellers, on="seller_id", how="left")

#  Step 4: Add products 
order_customer_item_seller_product = pd.merge(order_customer_item_seller, product, on="product_id", how="left")

''' Step 5: clean geolocation table by grouping rows by zip prefix + state and replacing
    the raw coordinates with the average latitude and longitude of each group
- why is that? : In the Olist dataset, the same zip-code area appears many times with slightly different lat and lng values
    We take the average of those coordinates so that each zip code has one representative point. 
    This makes the locations consistent and usable later when we calculate distances between customers sellers
 '''

clean_geo = ( 
    location.groupby(["geolocation_zip_code_prefix", "geolocation_state"], as_index=False)
       .agg({
           "geolocation_lat": "mean",
           "geolocation_lng": "mean"
       })
)
clean_geo = clean_geo.drop_duplicates(subset=["geolocation_zip_code_prefix"]) #cleaning duplicates

#  Step 6: Add Customer Geolocation 
order_customer_item_seller_product_geo = pd.merge(
    order_customer_item_seller_product,
    clean_geo,
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left",
    suffixes=("", "_cust")
)

#  Step 7: Add Seller Geolocation 
order_customer_item_seller_product_geo = pd.merge(
    order_customer_item_seller_product_geo,
    clean_geo,
    left_on="seller_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left",
    suffixes=("_cust", "_sell")
)

#  Step 8: Drop unnecessary duplicate columns 
order_customer_item_seller_product_geo.drop(columns=["geolocation_zip_code_prefix_cust", "geolocation_zip_code_prefix_sell"], inplace=True)


In [56]:
order_customer_item_seller_product_geo.shape #check

(113425, 35)

In [58]:
order_customer_item_seller_product_geo.to_csv("shipping_pupdated.csv",index=False) # move it so i can continue working the excel